In [ ]:
!pip install "huggingface-hub<1.0"
!pip install triton
!pip install xxhash

In [ ]:
!git clone https://github.com/Kaminyou/nano-vLLM-TPU.git
!cd nano-vLLM-TPU && git checkout temp/workable-tpu
from huggingface_hub import snapshot_download
!mkdir Qwen3-0.6B
snapshot_download(
    repo_id="Qwen/Qwen3-0.6B",
    local_dir="./Qwen3-0.6B",
    local_dir_use_symlinks=False,
    resume_download=True
)

fatal: destination path 'nano-vLLM-TPU' already exists and is not an empty directory.
Already on 'temp/workable-tpu'
Your branch is up to date with 'origin/temp/workable-tpu'.
mkdir: cannot create directory ‘Qwen3-0.6B’: File exists


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

'/content/Qwen3-0.6B'

In [ ]:
%%writefile nano_vllm_v3.py
# -*- coding: utf-8 -*-
from functools import lru_cache
import torch
import time
import os
import argparse
from torch import nn
import torch.nn.functional as F
import torch.distributed as dist
from glob import glob
from safetensors import safe_open
from dataclasses import dataclass
from transformers import Qwen3Config
from copy import copy
from enum import Enum, auto
from itertools import count
from collections import deque
import xxhash
import numpy as np
from transformers import AutoConfig
import pickle
from multiprocessing.synchronize import Event
from multiprocessing.shared_memory import SharedMemory
import atexit
from dataclasses import fields
from time import perf_counter
from tqdm.auto import tqdm
from transformers import AutoTokenizer
import torch.multiprocessing as mp
from random import seed, randint



class SiluAndMul(nn.Module):

    def __init__(self):
        super().__init__()

    #@torch.compile
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x, y = x.chunk(2, -1)
        return F.silu(x) * y


class RMSNorm(nn.Module):

    def __init__(
        self,
        hidden_size: int,
        eps: float = 1e-6,
    ) -> None:
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(hidden_size))

    #@torch.compile
    def rms_forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        orig_dtype = x.dtype
        x = x.float()
        var = x.pow(2).mean(dim=-1, keepdim=True)
        x.mul_(torch.rsqrt(var + self.eps))
        x = x.to(orig_dtype).mul_(self.weight)
        return x

    #@torch.compile
    def add_rms_forward(
        self,
        x: torch.Tensor,
        residual: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        orig_dtype = x.dtype
        x = x.float().add_(residual.float())
        residual = x.to(orig_dtype)
        var = x.pow(2).mean(dim=-1, keepdim=True)
        x.mul_(torch.rsqrt(var + self.eps))
        x = x.to(orig_dtype).mul_(self.weight)
        return x, residual

    def forward(
        self,
        x: torch.Tensor,
        residual: torch.Tensor | None = None,
    ) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:
        if residual is None:
            return self.rms_forward(x)
        else:
            return self.add_rms_forward(x, residual)


def divide(numerator, denominator):
    assert numerator % denominator == 0
    return numerator // denominator


class LinearBase(nn.Module):

    def __init__(
        self,
        input_size: int,
        output_size: int,
        bias: bool = False,
        tp_dim: int | None = None,
    ):
        super().__init__()
        self.tp_dim = tp_dim
        self.tp_rank = dist.get_rank()
        self.tp_size = dist.get_world_size()
        self.weight = nn.Parameter(torch.empty(output_size, input_size))
        self.weight.weight_loader = self.weight_loader
        if bias:
            self.bias = nn.Parameter(torch.empty(output_size))
            self.bias.weight_loader = self.weight_loader
        else:
            self.register_parameter("bias", None)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError


class ReplicatedLinear(LinearBase):

    def __init__(
        self,
        input_size: int,
        output_size: int,
        bias: bool = False,
    ):
        super().__init__(input_size, output_size, bias)

    def weight_loader(self, param: nn.Parameter, loaded_weight: torch.Tensor):
        param.data.copy_(loaded_weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.linear(x, self.weight, self.bias)


class ColumnParallelLinear(LinearBase):

    def __init__(
        self,
        input_size: int,
        output_size: int,
        bias: bool = False,
    ):
        tp_size = dist.get_world_size()
        super().__init__(input_size, divide(output_size, tp_size), bias, 0)

    def weight_loader(self, param: nn.Parameter, loaded_weight: torch.Tensor):
        param_data = param.data
        shard_size = param_data.size(self.tp_dim)
        start_idx = self.tp_rank * shard_size
        loaded_weight = loaded_weight.narrow(self.tp_dim, start_idx, shard_size)
        param_data.copy_(loaded_weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.linear(x, self.weight, self.bias)


class MergedColumnParallelLinear(ColumnParallelLinear):

    def __init__(
        self,
        input_size: int,
        output_sizes: list[int],
        bias: bool = False,
    ):
        self.output_sizes = output_sizes
        super().__init__(input_size, sum(output_sizes), bias)

    def weight_loader(self, param: nn.Parameter, loaded_weight: torch.Tensor, loaded_shard_id: int):
        param_data = param.data
        shard_offset = sum(self.output_sizes[:loaded_shard_id]) // self.tp_size
        shard_size = self.output_sizes[loaded_shard_id] // self.tp_size
        param_data = param_data.narrow(self.tp_dim, shard_offset, shard_size)
        loaded_weight = loaded_weight.chunk(self.tp_size, self.tp_dim)[self.tp_rank]
        param_data.copy_(loaded_weight)


class QKVParallelLinear(ColumnParallelLinear):

    def __init__(
        self,
        hidden_size: int,
        head_size: int,
        total_num_heads: int,
        total_num_kv_heads: int | None = None,
        bias: bool = False,
    ):
        tp_size = dist.get_world_size()
        total_num_kv_heads = total_num_kv_heads or total_num_heads
        self.head_size = head_size
        self.num_heads = divide(total_num_heads, tp_size)
        self.num_kv_heads = divide(total_num_kv_heads, tp_size)
        output_size = (total_num_heads + 2 * total_num_kv_heads) * self.head_size
        super().__init__(hidden_size, output_size, bias)

    def weight_loader(self, param: nn.Parameter, loaded_weight: torch.Tensor, loaded_shard_id: str):
        param_data = param.data
        assert loaded_shard_id in ["q", "k", "v"]
        if loaded_shard_id == "q":
            shard_size = self.num_heads * self.head_size
            shard_offset = 0
        elif loaded_shard_id == "k":
            shard_size = self.num_kv_heads * self.head_size
            shard_offset = self.num_heads * self.head_size
        else:
            shard_size = self.num_kv_heads * self.head_size
            shard_offset = self.num_heads * self.head_size + self.num_kv_heads * self.head_size
        param_data = param_data.narrow(self.tp_dim, shard_offset, shard_size)
        loaded_weight = loaded_weight.chunk(self.tp_size, self.tp_dim)[self.tp_rank]
        param_data.copy_(loaded_weight)


class RowParallelLinear(LinearBase):

    def __init__(
        self,
        input_size: int,
        output_size: int,
        bias: bool = False,
    ):
        tp_size = dist.get_world_size()
        super().__init__(divide(input_size, tp_size), output_size, bias, 1)

    def weight_loader(self, param: nn.Parameter, loaded_weight: torch.Tensor):
        param_data = param.data
        shard_size = param_data.size(self.tp_dim)
        start_idx = self.tp_rank * shard_size
        loaded_weight = loaded_weight.narrow(self.tp_dim, start_idx, shard_size)
        param_data.copy_(loaded_weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = F.linear(x, self.weight, self.bias if self.tp_rank == 0 else None)
        if self.tp_size > 1:
            dist.all_reduce(y)
        return y


def apply_rotary_emb(
    x: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
) -> torch.Tensor:
    x1, x2 = torch.chunk(x.float(), 2, dim=-1)
    y1 = x1 * cos - x2 * sin
    y2 = x2 * cos + x1 * sin
    return torch.cat((y1, y2), dim=-1).to(x.dtype)


class RotaryEmbedding(nn.Module):

    def __init__(
        self,
        head_size: int,
        rotary_dim: int,
        max_position_embeddings: int,
        base: float,
    ) -> None:
        super().__init__()
        self.head_size = head_size
        assert rotary_dim == head_size
        inv_freq = 1.0 / (base**(torch.arange(0, rotary_dim, 2, dtype=torch.float) / rotary_dim))
        t = torch.arange(max_position_embeddings, dtype=torch.float)
        freqs = torch.einsum("i,j -> ij", t, inv_freq)
        cos = freqs.cos()
        sin = freqs.sin()
        cache = torch.cat((cos, sin), dim=-1).unsqueeze_(1)
        self.register_buffer("cos_sin_cache", cache, persistent=False)

    #@torch.compile
    def forward(
        self,
        positions: torch.Tensor,
        query: torch.Tensor,
        key: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        cos_sin = self.cos_sin_cache[positions]
        cos, sin = cos_sin.chunk(2, dim=-1)
        query = apply_rotary_emb(query, cos, sin)
        key = apply_rotary_emb(key, cos, sin)
        return query, key


@lru_cache(1)
def get_rope(
    head_size: int,
    rotary_dim: int,
    max_position: int,
    base: float,
    rope_scaling: dict | None = None,
):
    assert rope_scaling is None
    rotary_emb = RotaryEmbedding(head_size, rotary_dim, max_position, base)
    return rotary_emb


class Sampler(nn.Module):

    def __init__(self):
        super().__init__()

    #@torch.compile
    def forward(self, logits: torch.Tensor, temperatures: torch.Tensor):
        logits = logits.float().div_(temperatures.unsqueeze(dim=1))
        probs = torch.softmax(logits, dim=-1)
        sample_tokens = probs.div_(torch.empty_like(probs).exponential_(1).clamp_min_(1e-10)).argmax(dim=-1)
        return sample_tokens


def default_weight_loader(param: nn.Parameter, loaded_weight: torch.Tensor):
    param.data.copy_(loaded_weight)


def load_model(model: nn.Module, path: str):
    packed_modules_mapping = getattr(model, "packed_modules_mapping", {})
    for file in glob(os.path.join(path, "*.safetensors")):
        with safe_open(file, "pt", "cpu") as f:
            for weight_name in f.keys():
                for k in packed_modules_mapping:
                    if k in weight_name:
                        v, shard_id = packed_modules_mapping[k]
                        param_name = weight_name.replace(k, v)
                        param = model.get_parameter(param_name)
                        weight_loader = getattr(param, "weight_loader")
                        weight_loader(param, f.get_tensor(weight_name), shard_id)
                        break
                else:
                    param = model.get_parameter(weight_name)
                    weight_loader = getattr(param, "weight_loader", default_weight_loader)
                    weight_loader(param, f.get_tensor(weight_name))


@dataclass
class Context:
    is_prefill: bool = False
    cu_seqlens_q: torch.Tensor | None = None
    cu_seqlens_k: torch.Tensor | None = None
    max_seqlen_q: int = 0
    max_seqlen_k: int = 0
    slot_mapping: torch.Tensor | None = None
    context_lens: torch.Tensor | None = None
    block_tables: torch.Tensor | None = None
    # new context
    batch_size: int = 0
    padded_seq_len: int = 0
    padding_mask: torch.Tensor | None = None

_CONTEXT = Context()

def get_context():
    return _CONTEXT

def set_context(is_prefill, cu_seqlens_q=None, cu_seqlens_k=None, max_seqlen_q=0, max_seqlen_k=0, slot_mapping=None, context_lens=None, block_tables=None, batch_size=0, padded_seq_len=0, padding_mask=None):
    global _CONTEXT
    _CONTEXT = Context(is_prefill, cu_seqlens_q, cu_seqlens_k, max_seqlen_q, max_seqlen_k, slot_mapping, context_lens, block_tables, batch_size, padded_seq_len, padding_mask)

def reset_context():
    global _CONTEXT
    _CONTEXT = Context()


class VocabParallelEmbedding(nn.Module):

    def __init__(
        self,
        num_embeddings: int,
        embedding_dim: int,
    ):
        super().__init__()
        self.tp_rank = dist.get_rank()
        self.tp_size = dist.get_world_size()
        assert num_embeddings % self.tp_size == 0
        self.num_embeddings = num_embeddings
        self.num_embeddings_per_partition = self.num_embeddings // self.tp_size
        self.vocab_start_idx = self.num_embeddings_per_partition * self.tp_rank
        self.vocab_end_idx = self.vocab_start_idx + self.num_embeddings_per_partition
        self.weight = nn.Parameter(torch.empty(self.num_embeddings_per_partition, embedding_dim))
        self.weight.weight_loader = self.weight_loader

    def weight_loader(self, param: nn.Parameter, loaded_weight: torch.Tensor):
        param_data = param.data
        shard_size = param_data.size(0)
        start_idx = self.tp_rank * shard_size
        loaded_weight = loaded_weight.narrow(0, start_idx, shard_size)
        param_data.copy_(loaded_weight)

    def forward(self, x: torch.Tensor):
        if self.tp_size > 1:
            mask = (x >= self.vocab_start_idx) & (x < self.vocab_end_idx)
            x = mask * (x - self.vocab_start_idx)
        y = F.embedding(x, self.weight)
        if self.tp_size > 1:
            y = mask.unsqueeze(1) * y
            dist.all_reduce(y)
        return y


class ParallelLMHead(VocabParallelEmbedding):

    def __init__(
        self,
        num_embeddings: int,
        embedding_dim: int,
        bias: bool = False,
    ):
        assert not bias
        super().__init__(num_embeddings, embedding_dim)

    def forward(self, x: torch.Tensor):
        context = get_context()
        if context.is_prefill:
            last_indices = context.cu_seqlens_q[1:] - 1
            x = x[last_indices].contiguous()
        logits = F.linear(x, self.weight)
        if self.tp_size > 1:
            all_logits = [torch.empty_like(logits) for _ in range(self.tp_size)] if self.tp_rank == 0 else None
            dist.gather(logits, all_logits, 0)
            logits = torch.cat(all_logits, -1) if self.tp_rank == 0 else None
        return logits



def store_kvcache(key, value, k_cache, v_cache, slot_mapping):
    """
    XLA-Optimized store_kvcache.
    Flattens the cache to 1D to avoid complex tuple indexing on TPU.
    """
    # 1. Flatten the cache view
    # k_cache original: [num_blocks, block_size, num_heads, head_dim]
    # k_flat: [total_slots, num_heads, head_dim]
    k_flat = k_cache.view(-1, k_cache.shape[2], k_cache.shape[3])
    v_flat = v_cache.view(-1, v_cache.shape[2], v_cache.shape[3])

    # 2. Store using flat indices
    # This lowers to a simple 'scatter' operation on TPU.
    k_flat.index_copy_(0, slot_mapping.long(), key)
    v_flat.index_copy_(0, slot_mapping.long(), value)


class Attention(nn.Module):

    def __init__(
        self,
        num_heads,
        head_dim,
        scale,
        num_kv_heads,
        block_size=256,
    ):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.scale = scale
        self.num_kv_heads = num_kv_heads
        self.block_size = block_size

        self.num_q_per_kv = self.num_heads // self.num_kv_heads

        # Caches are initialized empty
        self.k_cache = torch.tensor([])
        self.v_cache = torch.tensor([])

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor):
        context = get_context()

        # Guard to prevent warmup crash
        if self.k_cache.numel() > 0 and context.slot_mapping.numel() > 0:
            if context.is_prefill:
                # Filter out the padded tokens using our mask before caching
                flat_mask = context.padding_mask.view(-1)
                valid_k = k[flat_mask]
                valid_v = v[flat_mask]
                store_kvcache(valid_k, valid_v, self.k_cache, self.v_cache, context.slot_mapping)
            else:
                store_kvcache(k, v, self.k_cache, self.v_cache, context.slot_mapping)

        # Perform Attention
        if context.is_prefill:
            return self._forward_prefill(q, k, v, context)
        else:
            return self._forward_decode(q, context)

    def _forward_prefill(self, q, k, v, context):
        B = context.batch_size
        S = context.padded_seq_len

        # Reshape from 1D flattened (B*S, H, D) back to 2D (B, S, H, D)
        q = q.view(B, S, self.num_heads, self.head_dim)
        k = k.view(B, S, self.num_kv_heads, self.head_dim)
        v = v.view(B, S, self.num_kv_heads, self.head_dim)

        if self.num_q_per_kv > 1:
            k = k.repeat_interleave(self.num_q_per_kv, dim=2)
            v = v.repeat_interleave(self.num_q_per_kv, dim=2)

        q_b = q.transpose(1, 2)
        k_b = k.transpose(1, 2)
        v_b = v.transpose(1, 2)

        pad_mask = context.padding_mask.unsqueeze(1).unsqueeze(2)
        causal_mask = torch.tril(torch.ones((S, S), device=q.device, dtype=torch.bool)).unsqueeze(0).unsqueeze(0)
        attn_mask = causal_mask & pad_mask

        o_b = F.scaled_dot_product_attention(
            q_b, k_b, v_b,
            attn_mask=attn_mask,
            scale=self.scale,
            is_causal=False
        )

        o_seq = o_b.transpose(1, 2).contiguous()
        return o_seq.view(B * S, self.num_heads, self.head_dim)

    def _forward_decode(self, q, context):
        """
        Decode Logic (Token generation using PagedAttention)
        """
        batch_size = q.shape[0]

        # --- FIX: Simplify Query Reshaping ---
        # q comes in as (Batch, Num_Heads, 1, Head_Dim)
        # We just need to ensure it's in that shape.
        q_b = q.view(batch_size, self.num_heads, 1, self.head_dim)

        # --- GATHER K/V FROM PAGED CACHE ---
        bs_per_request = context.block_tables
        max_blocks = bs_per_request.shape[1]

        # Flatten block table to gather all blocks at once
        flat_blocks = bs_per_request.flatten().long()
        # Clamp to avoid -1 index error (padding)
        safe_blocks = flat_blocks.clamp(min=0)

        # Gather blocks from cache
        k_gathered = self.k_cache.index_select(0, safe_blocks)
        v_gathered = self.v_cache.index_select(0, safe_blocks)

        # Reshape to (Batch, Total_Context_Len, H_kv, D)
        total_context_len = max_blocks * self.block_size
        k_gathered = k_gathered.view(batch_size, total_context_len, self.num_kv_heads, self.head_dim)
        v_gathered = v_gathered.view(batch_size, total_context_len, self.num_kv_heads, self.head_dim)

        # Handle GQA (Repeat KV heads to match Q heads)
        if self.num_q_per_kv > 1:
            k_gathered = k_gathered.repeat_interleave(self.num_q_per_kv, dim=2)
            v_gathered = v_gathered.repeat_interleave(self.num_q_per_kv, dim=2)

        # Permute for SDPA: (Batch, Heads, SeqLen, Dim)
        k_b = k_gathered.permute(0, 2, 1, 3)
        v_b = v_gathered.permute(0, 2, 1, 3)

        # --- Create Mask ---
        # Mask out padding tokens (since block tables are padded to max_blocks)
        seq_indices = torch.arange(total_context_len, device=q.device).unsqueeze(0)
        mask = seq_indices < context.context_lens.unsqueeze(1)

        # (Batch, 1, 1, Total_Len)
        attn_mask = mask.unsqueeze(1).unsqueeze(2)

        # Run Attention
        o_b = F.scaled_dot_product_attention(
            q_b, k_b, v_b,
            attn_mask=attn_mask,
            scale=self.scale,
            is_causal=False
        )

        # Output: (Batch, Heads, 1, Dim) -> (Batch, Heads, Dim)
        return o_b.squeeze(2)

# MIGHT NEED MODIFICATION
# qwen3.py


class Qwen3Attention(nn.Module):

    def __init__(
        self,
        hidden_size: int,
        num_heads: int,
        num_kv_heads: int,
        max_position: int = 4096 * 32,
        head_dim: int | None = None,
        rms_norm_eps: float = 1e-06,
        qkv_bias: bool = False,
        rope_theta: float = 10000,
        rope_scaling: tuple | None = None,
    ) -> None:
        super().__init__()
        tp_size = dist.get_world_size()
        self.total_num_heads = num_heads
        assert self.total_num_heads % tp_size == 0
        self.num_heads = self.total_num_heads // tp_size
        self.total_num_kv_heads = num_kv_heads
        assert self.total_num_kv_heads % tp_size == 0
        self.num_kv_heads = self.total_num_kv_heads // tp_size
        self.head_dim = head_dim or hidden_size // self.total_num_heads
        self.q_size = self.num_heads * self.head_dim
        self.kv_size = self.num_kv_heads * self.head_dim
        self.scaling = self.head_dim ** -0.5
        self.qkv_bias = qkv_bias

        self.qkv_proj = QKVParallelLinear(
            hidden_size,
            self.head_dim,
            self.total_num_heads,
            self.total_num_kv_heads,
            bias=qkv_bias,
        )
        self.o_proj = RowParallelLinear(
            self.total_num_heads * self.head_dim,
            hidden_size,
            bias=False,
        )
        self.rotary_emb = get_rope(
            self.head_dim,
            rotary_dim=self.head_dim,
            max_position=max_position,
            base=rope_theta,
            # rope_scaling=rope_scaling,
            rope_scaling=None,
        )
        self.attn = Attention(
            self.num_heads,
            self.head_dim,
            self.scaling,
            self.num_kv_heads,
        )
        if not self.qkv_bias:
            self.q_norm = RMSNorm(self.head_dim, eps=rms_norm_eps)
            self.k_norm = RMSNorm(self.head_dim, eps=rms_norm_eps)

    def forward(
        self,
        positions: torch.Tensor,
        hidden_states: torch.Tensor,
    ) -> torch.Tensor:
        qkv = self.qkv_proj(hidden_states)
        q, k, v = qkv.split([self.q_size, self.kv_size, self.kv_size], dim=-1)
        q = q.view(-1, self.num_heads, self.head_dim)
        k = k.view(-1, self.num_kv_heads, self.head_dim)
        v = v.view(-1, self.num_kv_heads, self.head_dim)
        if not self.qkv_bias:
            q = self.q_norm(q)
            k = self.k_norm(k)
        q, k = self.rotary_emb(positions, q, k)
        o = self.attn(q, k, v)
        output = self.o_proj(o.flatten(1, -1))
        return output


class Qwen3MLP(nn.Module):

    def __init__(
        self,
        hidden_size: int,
        intermediate_size: int,
        hidden_act: str,
    ) -> None:
        super().__init__()
        self.gate_up_proj = MergedColumnParallelLinear(
            hidden_size,
            [intermediate_size] * 2,
            bias=False,
        )
        self.down_proj = RowParallelLinear(
            intermediate_size,
            hidden_size,
            bias=False,
        )
        assert hidden_act == "silu"
        self.act_fn = SiluAndMul()

    def forward(self, x):
        gate_up = self.gate_up_proj(x)
        x = self.act_fn(gate_up)
        x = self.down_proj(x)
        return x


class Qwen3DecoderLayer(nn.Module):

    def __init__(
        self,
        config: Qwen3Config,
    ) -> None:
        super().__init__()
        self.self_attn = Qwen3Attention(
            hidden_size=config.hidden_size,
            num_heads=config.num_attention_heads,
            num_kv_heads=config.num_key_value_heads,
            max_position=config.max_position_embeddings,
            rms_norm_eps=config.rms_norm_eps,
            qkv_bias=getattr(config, 'attention_bias', True),
            head_dim=getattr(config, 'head_dim', None),
            rope_theta=getattr(config, "rope_theta", 1000000),
            rope_scaling=getattr(config, "rope_scaling", None),
        )
        self.mlp = Qwen3MLP(
            hidden_size=config.hidden_size,
            intermediate_size=config.intermediate_size,
            hidden_act=config.hidden_act,
        )
        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(
        self,
        positions: torch.Tensor,
        hidden_states: torch.Tensor,
        residual: torch.Tensor | None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        if residual is None:
            hidden_states, residual = self.input_layernorm(hidden_states), hidden_states
        else:
            hidden_states, residual = self.input_layernorm(hidden_states, residual)
        hidden_states = self.self_attn(positions, hidden_states)
        hidden_states, residual = self.post_attention_layernorm(hidden_states, residual)
        hidden_states = self.mlp(hidden_states)
        return hidden_states, residual


class Qwen3Model(nn.Module):

    def __init__(
        self,
        config: Qwen3Config,
    ) -> None:
        super().__init__()
        self.embed_tokens = VocabParallelEmbedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([Qwen3DecoderLayer(config) for _ in range(config.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(
        self,
        input_ids: torch.Tensor,
        positions: torch.Tensor,
    ) -> torch.Tensor:
        hidden_states = self.embed_tokens(input_ids)
        residual = None
        for layer in self.layers:
            hidden_states, residual = layer(positions, hidden_states, residual)
        hidden_states, _ = self.norm(hidden_states, residual)
        return hidden_states


class Qwen3ForCausalLM(nn.Module):
    packed_modules_mapping = {
        "q_proj": ("qkv_proj", "q"),
        "k_proj": ("qkv_proj", "k"),
        "v_proj": ("qkv_proj", "v"),
        "gate_proj": ("gate_up_proj", 0),
        "up_proj": ("gate_up_proj", 1),
    }

    def __init__(
        self,
        config: Qwen3Config
    ) -> None:
        super().__init__()
        self.model = Qwen3Model(config)
        self.lm_head = ParallelLMHead(config.vocab_size, config.hidden_size)
        if config.tie_word_embeddings:
            self.lm_head.weight.data = self.model.embed_tokens.weight.data

    def forward(
        self,
        input_ids: torch.Tensor,
        positions: torch.Tensor,
    ) -> torch.Tensor:
        return self.model(input_ids, positions)

    def compute_logits(
        self,
        hidden_states: torch.Tensor,
    ) -> torch.Tensor:
        return self.lm_head(hidden_states)


@dataclass
class Config:
    model: str
    max_num_batched_tokens: int = 16384
    max_num_seqs: int = 512
    max_model_len: int = 4096
    gpu_memory_utilization: float = 0.9
    tensor_parallel_size: int = 1
    enforce_eager: bool = False
    hf_config: AutoConfig | None = None
    eos: int = -1
    kvcache_block_size: int = 256
    num_kvcache_blocks: int = -1

    def __post_init__(self):
        assert os.path.isdir(self.model)
        assert self.kvcache_block_size % 256 == 0
        assert 1 <= self.tensor_parallel_size <= 8
        self.hf_config = AutoConfig.from_pretrained(self.model)
        self.max_model_len = min(self.max_model_len, self.hf_config.max_position_embeddings)
        assert self.max_num_batched_tokens >= self.max_model_len

@dataclass
class SamplingParams:
    temperature: float = 1.0
    max_tokens: int = 64
    ignore_eos: bool = False

    def __post_init__(self):
        assert self.temperature > 1e-10, "greedy sampling is not permitted"

class SequenceStatus(Enum):
    WAITING = auto()
    RUNNING = auto()
    FINISHED = auto()


class Sequence:
    block_size = 256
    counter = count()

    def __init__(self, token_ids: list[int], sampling_params = SamplingParams()):
        self.seq_id = next(Sequence.counter)
        self.status = SequenceStatus.WAITING
        self.token_ids = copy(token_ids)
        self.last_token = token_ids[-1]
        self.num_tokens = len(self.token_ids)
        self.num_prompt_tokens = len(token_ids)
        self.num_cached_tokens = 0
        self.block_table = []
        self.temperature = sampling_params.temperature
        self.max_tokens = sampling_params.max_tokens
        self.ignore_eos = sampling_params.ignore_eos

    def __len__(self):
        return self.num_tokens

    def __getitem__(self, key):
        return self.token_ids[key]

    @property
    def is_finished(self):
        return self.status == SequenceStatus.FINISHED

    @property
    def num_completion_tokens(self):
        return self.num_tokens - self.num_prompt_tokens

    @property
    def prompt_token_ids(self):
        return self.token_ids[:self.num_prompt_tokens]

    @property
    def completion_token_ids(self):
        return self.token_ids[self.num_prompt_tokens:]

    @property
    def num_cached_blocks(self):
        return self.num_cached_tokens // self.block_size

    @property
    def num_blocks(self):
        return (self.num_tokens + self.block_size - 1) // self.block_size

    @property
    def last_block_num_tokens(self):
        return self.num_tokens - (self.num_blocks - 1) * self.block_size

    def block(self, i):
        assert 0 <= i < self.num_blocks
        return self.token_ids[i*self.block_size: (i+1)*self.block_size]

    def append_token(self, token_id: int):
        self.token_ids.append(token_id)
        self.last_token = token_id
        self.num_tokens += 1

    def __getstate__(self):
        return (self.num_tokens, self.num_prompt_tokens, self.num_cached_tokens, self.block_table,
                self.token_ids if self.num_completion_tokens == 0 else self.last_token)

    def __setstate__(self, state):
        self.num_tokens, self.num_prompt_tokens, self.num_cached_tokens, self.block_table = state[:-1]
        if self.num_completion_tokens == 0:
            self.token_ids = state[-1]
        else:
            self.last_token = state[-1]


class Block:

    def __init__(self, block_id):
        self.block_id = block_id
        self.ref_count = 0
        self.hash = -1
        self.token_ids = []

    def update(self, hash: int, token_ids: list[int]):
        self.hash = hash
        self.token_ids = token_ids

    def reset(self):
        self.ref_count = 1
        self.hash = -1
        self.token_ids = []


class BlockManager:

    def __init__(self, num_blocks: int, block_size: int):
        self.block_size = block_size
        self.blocks: list[Block] = [Block(i) for i in range(num_blocks)]
        self.hash_to_block_id: dict[int, int] = dict()
        self.free_block_ids: deque[int] = deque(range(num_blocks))
        self.used_block_ids: set[int] = set()

    @classmethod
    def compute_hash(cls, token_ids: list[int], prefix: int = -1):
        h = xxhash.xxh64()
        if prefix != -1:
            h.update(prefix.to_bytes(8, "little"))
        h.update(np.array(token_ids).tobytes())
        return h.intdigest()

    def _allocate_block(self, block_id: int) -> Block:
        block = self.blocks[block_id]
        assert block.ref_count == 0
        block.reset()
        self.free_block_ids.remove(block_id)
        self.used_block_ids.add(block_id)
        return self.blocks[block_id]

    def _deallocate_block(self, block_id: int) -> Block:
        assert self.blocks[block_id].ref_count == 0
        self.used_block_ids.remove(block_id)
        self.free_block_ids.append(block_id)

    def can_allocate(self, seq: Sequence) -> bool:
        return len(self.free_block_ids) >= seq.num_blocks

    def allocate(self, seq: Sequence):
        assert not seq.block_table
        h = -1
        cache_miss = False
        for i in range(seq.num_blocks):
            token_ids = seq.block(i)
            h = self.compute_hash(token_ids, h) if len(token_ids) == self.block_size else -1
            block_id = self.hash_to_block_id.get(h, -1)
            if block_id == -1 or self.blocks[block_id].token_ids != token_ids:
                cache_miss = True
            if cache_miss:
                block_id = self.free_block_ids[0]
                block = self._allocate_block(block_id)
            else:
                seq.num_cached_tokens += self.block_size
                if block_id in self.used_block_ids:
                    block = self.blocks[block_id]
                    block.ref_count += 1
                else:
                    block = self._allocate_block(block_id)
            if h != -1:
                block.update(h, token_ids)
                self.hash_to_block_id[h] = block_id
            seq.block_table.append(block_id)

    def deallocate(self, seq: Sequence):
        for block_id in reversed(seq.block_table):
            block = self.blocks[block_id]
            block.ref_count -= 1
            if block.ref_count == 0:
                self._deallocate_block(block_id)
        seq.num_cached_tokens = 0
        seq.block_table.clear()

    def can_append(self, seq: Sequence) -> bool:
        return len(self.free_block_ids) >= (len(seq) % self.block_size == 1)

    def may_append(self, seq: Sequence):
        block_table = seq.block_table
        last_block = self.blocks[block_table[-1]]
        if len(seq) % self.block_size == 1:
            assert last_block.hash != -1
            block_id = self.free_block_ids[0]
            self._allocate_block(block_id)
            block_table.append(block_id)
        elif len(seq) % self.block_size == 0:
            assert last_block.hash == -1
            token_ids = seq.block(seq.num_blocks-1)
            prefix = self.blocks[block_table[-2]].hash if len(block_table) > 1 else -1
            h = self.compute_hash(token_ids, prefix)
            last_block.update(h, token_ids)
            self.hash_to_block_id[h] = last_block.block_id
        else:
            assert last_block.hash == -1


class Scheduler:

    def __init__(self, config: Config):
        self.max_num_seqs = config.max_num_seqs
        self.max_num_batched_tokens = config.max_num_batched_tokens
        self.eos = config.eos
        self.block_manager = BlockManager(config.num_kvcache_blocks, config.kvcache_block_size)
        self.waiting: deque[Sequence] = deque()
        self.running: deque[Sequence] = deque()

    def is_finished(self):
        return not self.waiting and not self.running

    def add(self, seq: Sequence):
        self.waiting.append(seq)

    def schedule(self) -> tuple[list[Sequence], bool]:
        # prefill
        scheduled_seqs = []
        num_seqs = 0
        num_batched_tokens = 0
        while self.waiting and num_seqs < self.max_num_seqs:
            seq = self.waiting[0]
            if num_batched_tokens + len(seq) > self.max_num_batched_tokens or not self.block_manager.can_allocate(seq):
                break
            num_seqs += 1
            self.block_manager.allocate(seq)
            num_batched_tokens += len(seq) - seq.num_cached_tokens
            seq.status = SequenceStatus.RUNNING
            self.waiting.popleft()
            self.running.append(seq)
            scheduled_seqs.append(seq)
        if scheduled_seqs:
            return scheduled_seqs, True

        # decode
        while self.running and num_seqs < self.max_num_seqs:
            seq = self.running.popleft()
            while not self.block_manager.can_append(seq):
                if self.running:
                    self.preempt(self.running.pop())
                else:
                    self.preempt(seq)
                    break
            else:
                num_seqs += 1
                self.block_manager.may_append(seq)
                scheduled_seqs.append(seq)
        assert scheduled_seqs
        self.running.extendleft(reversed(scheduled_seqs))
        return scheduled_seqs, False

    def preempt(self, seq: Sequence):
        seq.status = SequenceStatus.WAITING
        self.block_manager.deallocate(seq)
        self.waiting.appendleft(seq)

    def postprocess(self, seqs: list[Sequence], token_ids: list[int]) -> list[bool]:
        for seq, token_id in zip(seqs, token_ids):
            seq.append_token(token_id)
            if (not seq.ignore_eos and token_id == self.eos) or seq.num_completion_tokens == seq.max_tokens:
                seq.status = SequenceStatus.FINISHED
                self.block_manager.deallocate(seq)
                self.running.remove(seq)


class ModelRunner:

    def __init__(self, config: Config, rank: int, event: Event | list[Event], mode: str):
        assert mode in ['tpu', 'gpu']

        self.config = config
        hf_config = config.hf_config
        self.block_size = config.kvcache_block_size
        self.enforce_eager = True  # NOTE: make this always true
        self.world_size = config.tensor_parallel_size
        self.rank = rank
        self.event = event
        self.mode = mode

        # NOTE: for colab
        dist.init_process_group("gloo", "tcp://localhost:2333", world_size=self.world_size, rank=rank)

        if self.mode == 'gpu':
            torch.cuda.set_device(rank)
            self.device = torch.device(f'cuda:{torch.cuda.current_device()}')
        elif self.mode == 'tpu':
            import torch_xla.core.xla_model as xm
            self.xm = xm
            self.device = xm.xla_device()
        else:
            raise ValueError('Support GPU and TPU only.')

        default_dtype = torch.get_default_dtype()
        torch.set_default_dtype(hf_config.torch_dtype)  # NOTE: torch.bfloat16
        torch.set_default_device(self.device)  # NOTE: might need to delete

        self.model = Qwen3ForCausalLM(hf_config)
        load_model(self.model, config.model)

        # NOTE: might need
        # self.model.eval()
        # self.model.requires_grad_(False)

        self.sampler = Sampler()
        self.warmup_model()
        self.allocate_kv_cache()

        torch.set_default_device("cpu")
        torch.set_default_dtype(default_dtype)

        if self.world_size > 1:
            if rank == 0:
                self.shm = SharedMemory(name="nanovllm", create=True, size=2**20)
                dist.barrier()
            else:
                dist.barrier()
                self.shm = SharedMemory(name="nanovllm")
                self.loop()

    def exit(self):
        if self.world_size > 1:
            self.shm.close()
            dist.barrier()
            if self.rank == 0:
                self.shm.unlink()

        if self.mode == 'gpu':
            torch.cuda.synchronize()
        elif self.mode == 'tpu':
            self.xm.wait_device_ops()
        else:
            raise ValueError
        dist.destroy_process_group()

    def loop(self):
        while True:
            method_name, args = self.read_shm()
            self.call(method_name, *args)
            if method_name == "exit":
                break

    def read_shm(self):
        assert self.world_size > 1 and self.rank > 0
        self.event.wait()
        n = int.from_bytes(self.shm.buf[0:4], "little")
        method_name, *args = pickle.loads(self.shm.buf[4:n+4])
        self.event.clear()
        return method_name, args

    def write_shm(self, method_name, *args):
        assert self.world_size > 1 and self.rank == 0
        data = pickle.dumps([method_name, *args])
        n = len(data)
        self.shm.buf[0:4] = n.to_bytes(4, "little")
        self.shm.buf[4:n+4] = data
        for event in self.event:
            event.set()

    def call(self, method_name, *args):
        if self.world_size > 1 and self.rank == 0:
            self.write_shm(method_name, *args)
        method = getattr(self, method_name, None)
        return method(*args)

    def warmup_model(self):
        if self.mode == 'gpu':
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        max_num_batched_tokens, max_model_len = self.config.max_num_batched_tokens, self.config.max_model_len
        num_seqs = min(max_num_batched_tokens // max_model_len, self.config.max_num_seqs)
        seqs = [Sequence([0] * max_model_len) for _ in range(num_seqs)]
        self.run(seqs, True)

        if self.mode == 'gpu':
            torch.cuda.empty_cache()
        elif self.mode == 'tpu':
            # NOTE: not sure if needed
            self.xm.mark_step()
            pass
        else:
            raise ValueError
        print('finish warmup')

    def allocate_kv_cache(self):
        config = self.config
        hf_config = config.hf_config

        if self.mode == 'gpu':
            free, total = torch.cuda.mem_get_info()
            used = total - free
            peak = torch.cuda.memory_stats()["allocated_bytes.all.peak"]
            current = torch.cuda.memory_stats()["allocated_bytes.all.current"]
            num_kv_heads = hf_config.num_key_value_heads // self.world_size
            head_dim = getattr(hf_config, "head_dim", hf_config.hidden_size // hf_config.num_attention_heads)
            block_bytes = 2 * hf_config.num_hidden_layers * self.block_size * num_kv_heads * head_dim * hf_config.torch_dtype.itemsize
            config.num_kvcache_blocks = int(total * config.gpu_memory_utilization - used - peak + current) // block_bytes
            print(f'total: {total / 1024 / 1024 :.3f}MB')
            print(f'used: {used / 1024 / 1024 :.3f}MB')
            print(f'peak: {peak / 1024 / 1024 :.3f}MB')
            print(f'current: {current / 1024 / 1024 :.3f}MB')
            print(f'block size: {block_bytes / 1024 / 1024 :.3f}MB')
            print(f'# of blocks: {config.num_kvcache_blocks}')
        elif self.mode == 'tpu':
            mem_info = self.xm.get_memory_info(self.device)
            total = mem_info['bytes_limit']  # NOTE: TPU API
            used = mem_info['bytes_used']  # NOTE: TPU API

            num_kv_heads = hf_config.num_key_value_heads // self.world_size
            head_dim = getattr(hf_config, "head_dim", hf_config.hidden_size // hf_config.num_attention_heads)
            block_bytes = 2 * hf_config.num_hidden_layers * self.block_size * num_kv_heads * head_dim * hf_config.torch_dtype.itemsize
            # config.num_kvcache_blocks = int(total * config.gpu_memory_utilization - used) // block_bytes  # NOTE: this doesn't work
            # config.num_kvcache_blocks = int(total * 0.5 - used) // block_bytes  # NOTE: this work (=500)


            config.num_kvcache_blocks = 168  # set the same number with T4 GPU
            print(f'total: {total / 1024 / 1024 :.3f}MB')
            print(f'used: {used / 1024 / 1024 :.3f}MB')
            print(f'block size: {block_bytes / 1024 / 1024 :.3f}MB')
            print(f'# of blocks: {config.num_kvcache_blocks}')
        else:
            raise ValueError

        assert config.num_kvcache_blocks > 0
        # self.kv_cache = torch.empty(2, hf_config.num_hidden_layers, config.num_kvcache_blocks, self.block_size, num_kv_heads, head_dim)
        # layer_id = 0
        # for module in self.model.modules():
        #     if hasattr(module, "k_cache") and hasattr(module, "v_cache"):
        #         module.k_cache = self.kv_cache[0, layer_id]
        #         module.v_cache = self.kv_cache[1, layer_id]
        #         layer_id += 1
        for module in self.model.modules():
            if hasattr(module, "k_cache") and hasattr(module, "v_cache"):
                module.k_cache = torch.empty(
                    config.num_kvcache_blocks,
                    self.block_size,
                    num_kv_heads,
                    head_dim
                )
                module.v_cache = torch.empty(
                    config.num_kvcache_blocks,
                    self.block_size,
                    num_kv_heads,
                    head_dim
                )

    def prepare_block_tables(self, seqs: list[Sequence]):
        max_len = max(len(seq.block_table) for seq in seqs)
        block_tables = [seq.block_table + [-1] * (max_len - len(seq.block_table)) for seq in seqs]
        #block_tables = torch.tensor(block_tables, dtype=torch.int32, pin_memory=True).cuda(non_blocking=True)
        block_tables = torch.tensor(block_tables, dtype=torch.int32).to(self.device)
        return block_tables

    def prepare_prefill(self, seqs: list[Sequence]):
        batch_size = len(seqs)
        max_seq_len = max(len(seq) - seq.num_cached_tokens for seq in seqs)

        bucket_len = ((max_seq_len + 127) // 128) * 128
        input_ids_2d = torch.zeros((batch_size, bucket_len), dtype=torch.int64)
        positions_2d = torch.zeros((batch_size, bucket_len), dtype=torch.int64)
        padding_mask = torch.zeros((batch_size, bucket_len), dtype=torch.bool)
        slot_mapping = []
        cu_seqlens_q = [0]

        for i, seq in enumerate(seqs):
            seqlen = len(seq)
            actual_len = seqlen - seq.num_cached_tokens
            tokens = seq[seq.num_cached_tokens:]
            input_ids_2d[i, :actual_len] = torch.tensor(tokens, dtype=torch.int64)
            positions_2d[i, :actual_len] = torch.tensor(list(range(seq.num_cached_tokens, seqlen)), dtype=torch.int64)
            padding_mask[i, :actual_len] = True

            # Point to the last valid token of this sequence in the flattened B*S array
            cu_seqlens_q.append(i * bucket_len + actual_len)

            if not seq.block_table:
                continue

            for j in range(seq.num_cached_blocks, seq.num_blocks):
                start = seq.block_table[j] * self.block_size
                end = start + self.block_size if j != seq.num_blocks - 1 else start + seq.last_block_num_tokens
                slot_mapping.extend(list(range(start, end)))

        # FLATTEN TO 1D for the rest of the model
        input_ids = input_ids_2d.view(-1).to(self.device)
        positions = positions_2d.view(-1).to(self.device)
        padding_mask = padding_mask.to(self.device)
        slot_mapping = torch.tensor(slot_mapping, dtype=torch.int32).to(self.device)
        cu_seqlens_q = torch.tensor(cu_seqlens_q, dtype=torch.int32).to(self.device)

        set_context(
            True,
            cu_seqlens_q=cu_seqlens_q,
            slot_mapping=slot_mapping,
            batch_size=batch_size,
            padded_seq_len=bucket_len,
            padding_mask=padding_mask
        )
        return input_ids, positions

    def prepare_decode(self, seqs: list[Sequence]):
        input_ids = []
        positions = []
        slot_mapping = []
        context_lens = []
        for seq in seqs:
            input_ids.append(seq.last_token)
            positions.append(len(seq) - 1)
            context_lens.append(len(seq))
            slot_mapping.append(seq.block_table[-1] * self.block_size + seq.last_block_num_tokens  - 1)
        input_ids = torch.tensor(input_ids, dtype=torch.int64).to(self.device)
        positions = torch.tensor(positions, dtype=torch.int64).to(self.device)
        slot_mapping = torch.tensor(slot_mapping, dtype=torch.int32).to(self.device)
        context_lens = torch.tensor(context_lens, dtype=torch.int32).to(self.device)
        block_tables = self.prepare_block_tables(seqs)
        set_context(False, slot_mapping=slot_mapping, context_lens=context_lens, block_tables=block_tables)
        return input_ids, positions

    def prepare_sample(self, seqs: list[Sequence]):
        temperatures = []
        for seq in seqs:
            temperatures.append(seq.temperature)
        temperatures = torch.tensor(temperatures, dtype=torch.float32).to(self.device)
        return temperatures

    @torch.no_grad()
    def run_model(self, input_ids: torch.Tensor, positions: torch.Tensor, is_prefill: bool):
        logits = self.model.compute_logits(self.model(input_ids, positions))
        if self.mode == 'tpu':
            self.xm.mark_step() # NOTE: if needed?
        return logits

    def run(self, seqs: list[Sequence], is_prefill: bool) -> list[int]:
        input_ids, positions = self.prepare_prefill(seqs) if is_prefill else self.prepare_decode(seqs)
        temperatures = self.prepare_sample(seqs) if self.rank == 0 else None
        logits = self.run_model(input_ids, positions, is_prefill)
        token_ids = self.sampler(logits, temperatures).tolist() if self.rank == 0 else None
        reset_context()
        return token_ids



class LLMEngine:

    def __init__(self, model, mode, **kwargs):
        config_fields = {field.name for field in fields(Config)}
        config_kwargs = {k: v for k, v in kwargs.items() if k in config_fields}
        config = Config(model, **config_kwargs)
        self.ps = []
        self.events = []
        ctx = mp.get_context("spawn")
        for i in range(1, config.tensor_parallel_size):
            event = ctx.Event()
            process = ctx.Process(target=ModelRunner, args=(config, i, event))
            process.start()
            self.ps.append(process)
            self.events.append(event)
        self.model_runner = ModelRunner(config, 0, self.events, mode=mode)
        self.tokenizer = AutoTokenizer.from_pretrained(config.model, use_fast=True)
        config.eos = self.tokenizer.eos_token_id
        self.scheduler = Scheduler(config)
        atexit.register(self.exit)

    def exit(self):
        self.model_runner.call("exit")
        del self.model_runner
        for p in self.ps:
            p.join()

    def add_request(self, prompt: str | list[int], sampling_params: SamplingParams):
        if isinstance(prompt, str):
            prompt = self.tokenizer.encode(prompt)
        seq = Sequence(prompt, sampling_params)
        self.scheduler.add(seq)

    def step(self):
        seqs, is_prefill = self.scheduler.schedule()
        token_ids = self.model_runner.call("run", seqs, is_prefill)
        self.scheduler.postprocess(seqs, token_ids)
        outputs = [(seq.seq_id, seq.completion_token_ids) for seq in seqs if seq.is_finished]
        num_tokens = sum(len(seq) for seq in seqs) if is_prefill else -len(seqs)
        return outputs, num_tokens

    def is_finished(self):
        return self.scheduler.is_finished()

    def generate(
        self,
        prompts: list[str] | list[list[int]],
        sampling_params: SamplingParams | list[SamplingParams],
        use_tqdm: bool = True,
    ) -> tuple[list[dict], dict]:
        if use_tqdm:
            pbar = tqdm(total=len(prompts), desc="Generating", dynamic_ncols=True)
        if not isinstance(sampling_params, list):
            sampling_params = [sampling_params] * len(prompts)
        for prompt, sp in zip(prompts, sampling_params):
            self.add_request(prompt, sp)

        outputs = {}

        # --- NEW: Track cumulative metrics ---
        total_prefill_tokens = 0
        total_prefill_time = 0.0
        total_decode_tokens = 0
        total_decode_time = 0.0

        while not self.is_finished():
            t = perf_counter()
            output, num_tokens = self.step()
            elapsed = perf_counter() - t

            if num_tokens > 0:
                # Prefill phase
                total_prefill_tokens += num_tokens
                total_prefill_time += elapsed
                current_prefill_tp = num_tokens / elapsed if elapsed > 0 else 0
                if use_tqdm:
                    pbar.set_postfix({"Prefill": f"{int(current_prefill_tp)}tok/s"})
            else:
                # Decode phase (num_tokens is negative)
                dec_tokens = -num_tokens
                total_decode_tokens += dec_tokens
                total_decode_time += elapsed
                current_decode_tp = dec_tokens / elapsed if elapsed > 0 else 0
                if use_tqdm:
                    pbar.set_postfix({"Decode": f"{int(current_decode_tp)}tok/s"})

            for seq_id, token_ids in output:
                outputs[seq_id] = token_ids
                if use_tqdm:
                    pbar.update(1)

        outputs = [outputs[seq_id] for seq_id in sorted(outputs.keys())]
        outputs = [{"text": self.tokenizer.decode(token_ids), "token_ids": token_ids} for token_ids in outputs]
        if use_tqdm:
            pbar.close()

        # --- NEW: Return the metrics dictionary ---
        metrics = {
            "prefill_tokens": total_prefill_tokens,
            "prefill_time": total_prefill_time,
            "decode_tokens": total_decode_tokens,
            "decode_time": total_decode_time
        }

        return outputs, metrics

# !!!IMPORTANT!!!!
# THOSE SHOULD NOT BE MODIFIED
class LLM(LLMEngine):
    pass


import os
import argparse
from random import seed, randint

def main():
    parser = argparse.ArgumentParser(description="Benchmark Nano vLLM")
    parser.add_argument("--mode", type=str, required=True, choices=["gpu", "tpu"], help="Hardware mode: 'gpu' or 'tpu'")
    parser.add_argument("--model_path", type=str, default="./Qwen3-0.6B/", help="Path to the downloaded model")
    parser.add_argument("--tp_size", type=int, default=1, help="Tensor parallel size")
    parser.add_argument("--num_seqs", type=int, default=50, help="Total number of sequences to generate")
    parser.add_argument("--batch_size", type=int, default=10, help="Number of sequences per iteration (max 50)")
    args = parser.parse_args()

    if args.batch_size > 50:
        print("Warning: batch_size exceeds the maximum limit of 50. Clamping to 50.")
        args.batch_size = 50

    seed(0)
    num_seqs = args.num_seqs
    max_input_len = 256
    max_output_len = 256

    path = os.path.expanduser(args.model_path)

    print(f"--- Initializing LLM Engine in {args.mode.upper()} mode ---")

    llm = LLM(path, mode=args.mode, enforce_eager=True, tensor_parallel_size=args.tp_size, max_model_len=4096)

    prompt_token_ids = [[randint(0, 10000) for _ in range(randint(100, max_input_len))] for _ in range(num_seqs)]
    sampling_params = [SamplingParams(temperature=0.6, ignore_eos=True, max_tokens=randint(100, max_output_len)) for _ in range(num_seqs)]

    print("Running warmup benchmark...")
    # Capture and ignore the metrics for the warmup run
    _, _ = llm.generate([[0, 1, 2, 3]], [SamplingParams(max_tokens=1)], use_tqdm=False)

    print(f"Starting benchmark with {num_seqs} total sequences, chunked into batches of {args.batch_size}...")

    # Global trackers for the final average
    overall_prefill_tokens = 0
    overall_prefill_time = 0.0
    overall_decode_tokens = 0
    overall_decode_time = 0.0

    for i in range(0, num_seqs, args.batch_size):
        batch_prompts = prompt_token_ids[i:i + args.batch_size]
        batch_params = sampling_params[i:i + args.batch_size]

        iteration = (i // args.batch_size) + 1
        print(f"\n--- Iteration {iteration} ({len(batch_prompts)} sequences) ---")

        # Unpack the tuple returned by our updated generate() method
        outputs, metrics = llm.generate(batch_prompts, batch_params, use_tqdm=True)

        # Accumulate metrics
        overall_prefill_tokens += metrics["prefill_tokens"]
        overall_prefill_time += metrics["prefill_time"]
        overall_decode_tokens += metrics["decode_tokens"]
        overall_decode_time += metrics["decode_time"]

        # Calculate iteration throughputs safely
        p_tp = metrics["prefill_tokens"] / metrics["prefill_time"] if metrics["prefill_time"] > 0 else 0
        d_tp = metrics["decode_tokens"] / metrics["decode_time"] if metrics["decode_time"] > 0 else 0

        print(f"Iteration {iteration} -> Prefill: {p_tp:.2f} tok/s | Decode: {d_tp:.2f} tok/s")

    # Calculate overall throughputs safely
    final_p_tp = overall_prefill_tokens / overall_prefill_time if overall_prefill_time > 0 else 0
    final_d_tp = overall_decode_tokens / overall_decode_time if overall_decode_time > 0 else 0

    print("\n===================================================")
    print(f"Overall Prefill Throughput: {final_p_tp:.2f} tok/s ({overall_prefill_tokens} tok / {overall_prefill_time:.2f} s)")
    print(f"Overall Decode Throughput:  {final_d_tp:.2f} tok/s ({overall_decode_tokens} tok / {overall_decode_time:.2f} s)")
    print("===================================================")

    # if hasattr(llm, 'exit'):
    #     llm.exit()

if __name__ == "__main__":
    main()

Overwriting nano_vllm_v3.py


In [ ]:
!python nano_vllm_v3.py --mode tpu --num_seqs 100 --batch_size 10

--- Initializing LLM Engine in TPU mode ---
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
/content/nano_vllm_v3.py:1088: DeprecationWarning: Use torch_xla.device instead
  self.device = xm.xla_device()
`torch_dtype` is deprecated! Use `dtype` instead!
/content/nano_vllm_v3.py:1328: DeprecationWarning: Use torch_xla.sync instead
  self.xm.mark_step() # NOTE: if needed?
/content/nano_vllm_v3.py:1177: DeprecationWarning: Use torch_xla.sync instead
  self.xm.mark_step()
finish warmup
total: 16126.000MB
used: 1883.978MB
block size: 28.000MB
# of blocks: 168
Running warmup benchmark...
Starting benchmark with 100 total sequences, chunked into batches of 10...

--- Iteration 1 (10 sequences) ---
Generating: 100% 10/10 [01:30<00:00,  9.05s/it, Decode=7tok/s]
Iteration 1 -> Prefill: 202.49 tok/s | Decode: 25.20 tok/s

--- Iteration 2 (10 sequences) ---
Generating: 100% 10/10 [00:36<00:00,  3.65s/it, Decode=8tok/s]
Iteration 2 -> Prefill: 274.50 tok/s

In [ ]:
# !python nano_vllm_v3.py --mode gpu --num_seqs 100 --batch_size 10